# Weak supervision · data besar tanpa label
**Untuk kasus: ratusan ribu teks, label tidak ada atau sangat sedikit, tetapi tetap harus mengeluarkan kelas untuk setiap baris.**

Alurnya:

1. **Labeling function (LF):** beberapa sumber label lemah. Setiap LF boleh abstain kalau tidak yakin.
2. **Label model:** gabungan semua LF. Bobot setiap LF diperkirakan dari kesepakatannya dengan LF lain, tanpa label.
3. **End model:** classifier yang dilatih dari label lemah, sehingga bisa memberi kelas pada baris yang tidak tersentuh LF dan menggeneralisasi lebih jauh.

| Jenis | Sumber | Murah? |
|---|---|---|
| LF kata kunci | leksikon positif/negatif + penanganan negasi ("tidak bagus" → negatif) | ya, semua baris |
| LF pertanyaan | tanda tanya dan kata tanya, tanpa kata sentimen → netral | ya |
| LF emoji | emoji positif/negatif | ya |
| LF e5 prototipe | kemiripan embedding e5 frozen ke deskripsi label | ya, sekitar 1 menit untuk 135 ribu teks |
| LF zero-shot NLI | mDeBERTa-v3 MNLI-XNLI, hanya di sampel `N_NLI` | mahal, jadi hanya sampel |
| End model | mayoritas, LF kata kunci, TF-IDF+LR, e5+LR, **IndoBERT fine-tune dari label lemah**, ensemble | |

**Data contoh:** [ulasan Gojek](https://www.kaggle.com/datasets/ucupsedaya/gojek-app-reviews-bahasa-indonesia), 225 ribu baris. Rating bintang disembunyikan sebagai **audit**: 1–2 = negatif, 3 = netral, 4–5 = positif. Rating adalah sinyal manusia asli, tapi tetap berisik, misalnya bintang 5 yang isinya keluhan.

`N_DEV = 0` berarti tanpa label sama sekali. `N_DEV > 0` mensimulasikan panitia yang memberi sedikit label: label itu dipakai untuk menimbang LF dan memilih model, dan barisnya dikeluarkan dari audit.

## 00 · Konfigurasi

Parameter papermill. Untuk kasus MBG, ganti preset, deskripsi label, dan kata kunci domain. Label boleh lebih dari tiga, dan setiap label cukup diberi deskripsi serta kata kunci.

In [1]:
DATASET = "gojek"
SEED, TOP = 42, 10
N_DEV = 0                          # 0 = tanpa label; misal 500 = panitia memberi 500 label
N_NLI, N_BERT = 20000, 30000       # sampel untuk zero-shot NLI dan untuk fine-tune IndoBERT
E5_MARGIN, NLI_MIN, CONF_MIN = .02, .50, .60
BERT_EPOCHS, BERT_LEN, BATCH = 2, 64, 256
BUDGET_MINUTES = 290
LABELS = ["negatif", "netral", "positif"]
DESCRIPTIONS = {"negatif": ["keluhan, kecewa, aplikasi buruk dan bermasalah", "pelayanan jelek, lambat, mahal, mengecewakan"],
                "netral": ["pertanyaan atau informasi tanpa penilaian", "bertanya cara memakai fitur atau meminta informasi"],
                "positif": ["pujian, puas, aplikasi bagus dan membantu", "pelayanan baik, cepat, ramah, memuaskan"]}
HYPOTHESES = {"negatif": "Teks ini berisi keluhan atau penilaian negatif.",
              "netral": "Teks ini netral, berisi pertanyaan atau informasi tanpa penilaian.",
              "positif": "Teks ini berisi pujian atau penilaian positif."}
PRESETS = {"gojek": dict(file="GojekAppReviewV4.0.0-V4.9.3_Cleaned.csv", text="content", audit="score",
                         audit_map={1: "negatif", 2: "negatif", 3: "netral", 4: "positif", 5: "positif"},
                         pos_extra="driver ramah cepat sampai tepat waktu", neg_extra="cancel batal saldo hilang topup gagal ongkir"),
           "mbg": dict(file="comment_sentiment.csv", text="Comment", audit="Sentiment",
                       audit_map={"negative": "negatif", "neutral": "netral", "positive": "positif"},
                       pos_extra="bergizi bermanfaat dukung mendukung setuju lanjutkan", neg_extra="korupsi koruptor keracunan basi tolak hentikan pemborosan")}

In [2]:
# Parameters
N_DEV = 500


In [3]:
from pathlib import Path
P = PRESETS[DATASET]; FILE, TEXT_COL, AUDIT_COL = P["file"], P["text"], P["audit"]
OUT = Path(f"/kaggle/working/hasil_weak_{DATASET}_dev{N_DEV}")
K = len(LABELS); LID = {name: i for i, name in enumerate(LABELS)}
print("dataset:", DATASET, "| label:", LABELS, "| N_DEV:", N_DEV)

dataset: gojek | label: ['negatif', 'netral', 'positif'] | N_DEV: 500


## 01 · Lingkungan Kaggle & pencatat waktu

Model diambil dari paket offline kalau ada, dan dari Hugging Face kalau tidak. e5 dan mDeBERTa hanya dipakai inferensi (frozen). IndoBERT di-fine-tune dari label lemah.

In [4]:
import sys, os, subprocess
assert Path("/kaggle/input").exists(), "Jalankan notebook ini di Kaggle."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ftfy>=6.3,<7", "emoji>=2.14,<3", "langid==1.1.6", "sentencepiece>=0.2,<1"], check=True)
os.environ.update(TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1", USE_TF="0", USE_FLAX="0")

In [5]:
import json, time, random, re, html, unicodedata, warnings
from collections import Counter
from contextlib import contextmanager
import numpy as np, pandas as pd, torch, emoji, ftfy
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT.mkdir(exist_ok=True); INPUT = Path("/kaggle/input"); RUN_START = time.perf_counter(); TIMES = []
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada")

@contextmanager
def stage(name, n=None):
    start = time.perf_counter(); yield
    TIMES.append({"tahap": name, "detik": time.perf_counter() - start, "n": n})
    print(f"[waktu] {name}: {TIMES[-1]['detik']:.0f} detik | total {(time.perf_counter() - RUN_START) / 60:.1f} menit")
def show(title, table):
    print("\n" + title); display((table.to_frame() if isinstance(table, pd.Series) else table).round(3))
def distribution(values):
    n = values.value_counts(dropna=False); return pd.DataFrame({"n": n, "%": n / (len(values) or np.nan) * 100})
def locate(name, hf_id):
    found = list(INPUT.glob(f"**/models/{name}/config.json")); return str(found[0].parent) if found else hf_id
pd.set_option("display.max_columns", None, "display.max_colwidth", 100, "display.width", 200)

GPU: Tesla T4


## 02 · Muat data, dedupe, dan pisahkan audit

Semua model jalan di **teks unik** `U`, dengan `count` sebagai bobot, jadi skor yang dilaporkan setara skor per baris. Label audit dihitung per teks unik dengan suara terbanyak, lalu disembunyikan di `U["_audit"]`. Kalau `N_DEV > 0`, sejumlah teks unik diambil sebagai dev berlabel.

In [6]:
with stage("muat_data"):
    path = next(INPUT.rglob(FILE)); raw = pd.read_csv(path, low_memory=False)
text = raw[TEXT_COL]
quality = pd.DataFrame({"teks_null": text.isna(), "teks_non_string": text.notna() & ~text.map(lambda x: isinstance(x, str)),
    "teks_kosong": text.fillna("").astype(str).str.strip().eq("")})
show("Kualitas input", pd.DataFrame({"n": quality.sum(), "%": quality.mean() * 100}))
data = raw.loc[~quality.any(axis=1)].reset_index(names="row_id")
data["key"] = data[TEXT_COL].astype(str).str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
data["_audit"] = data[AUDIT_COL].map(P["audit_map"]) if AUDIT_COL in data else None
U = data.groupby("key", sort=False).agg(text=(TEXT_COL, "first"), count=("key", "size"),
    _audit=("_audit", lambda s: s.mode().iloc[0] if s.notna().any() else None)).reset_index()
W = U["count"].to_numpy()
rng = np.random.default_rng(SEED)
DEV = np.sort(rng.choice(len(U), N_DEV, replace=False)) if N_DEV else np.array([], dtype=int)
is_dev = np.isin(np.arange(len(U)), DEV)
show("Ukuran", pd.Series({"baris": len(data), "teks_unik": len(U), "baris_di_teks_berulang_%": 100 * W[W > 1].sum() / len(data),
    "dev_berlabel": len(DEV), "teks_terbanyak": U.text[W.argmax()], "n_terbanyak": W.max()}))

[waktu] muat_data: 1 detik | total 0.0 menit

Kualitas input


,n,%
teks_null,2,0.001
teks_non_string,0,0.000
teks_kosong,79,0.035



Ukuran


,0
baris,224923
teks_unik,134606
baris_di_teks_berulang_%,42.565678
dev_berlabel,500
teks_terbanyak,Mantap
n_terbanyak,9924


## 03 · EDA wajib weak supervision · angka

Yang paling menentukan di weak supervision: seberapa pendek teksnya (kata kunci jarang kena), seberapa sering ada negasi (kata kunci bisa terbalik), emoji (sumber LF gratis), bahasa campuran (leksikon Indonesia meleset), dan teks berulang (satu kesalahan LF terulang ribuan kali). Distribusi label audit **tidak** ditampilkan di sini, supaya tidak memengaruhi rancangan LF.

In [7]:
with stage("eda", len(U)):
    t = U.text.astype(str); words = t.str.split().str.len()
    show("3.1 Panjang (berbobot baris)", pd.Series({"kata_median": np.median(np.repeat(words, W)), "<=2_kata_%": 100 * W[words <= 2].sum() / W.sum(),
        "<=5_kata_%": 100 * W[words <= 5].sum() / W.sum(), ">30_kata_%": 100 * W[words > 30].sum() / W.sum()}))
    flags = pd.DataFrame({"negasi": t.str.contains(r"\b(?:tidak|tak|gak|ga|nggak|enggak|bukan|belum|kurang|jangan)\b", case=False),
        "tanda_tanya": t.str.contains(r"\?"), "emoji": t.map(lambda s: emoji.emoji_count(s) > 0),
        "kata_inggris": t.str.contains(r"\b(?:good|nice|best|bad|the|is|very|thanks|please)\b", case=False),
        "slang": t.str.contains(r"\b(?:gk|gak|ga|bgt|yg|udh|klo|jg|tdk|mantul|mantab)\b", case=False),
        "huruf_ulang": t.str.contains(r"([^\W\d_])\1{2,}"), "angka": t.str.contains(r"\d")})
    show("3.2 Kondisi teks · % baris", (flags.mul(W, axis=0).sum() / W.sum() * 100).sort_values(ascending=False))
    seqs = t.str.casefold().str.findall(r"[^\W\d_]+")
    counts = pd.Series(Counter(w for s, c in zip(seqs, W) for w in s), dtype="int64").sort_values(ascending=False)
    show("3.3 Kata teratas (berbobot baris) · bahan leksikon", counts.head(25))
    show("Teks paling sering · satu kesalahan LF di sini terulang ribuan kali", U.nlargest(TOP, "count")[["text", "count"]])


3.1 Panjang (berbobot baris)


,0
kata_median,3.000
<=2_kata_%,42.974
<=5_kata_%,61.814
>30_kata_%,6.051



3.2 Kondisi teks · % baris


,0
slang,17.825
negasi,16.106
angka,10.123
kata_inggris,6.517
huruf_ulang,4.853
emoji,0.000
tanda_tanya,0.000



3.3 Kata teratas (berbobot baris) · bahan leksikon


,0
saya,33063
di,32942
gojek,30728
bisa,23547
dan,22651
nya,21860
aplikasi,19881
ada,18588
driver,18485
yg,15888



Teks paling sering · satu kesalahan LF di sini terulang ribuan kali


,text,count
4,Mantap,9924
7,Ok,9850
91,Good,6894
5,Bagus,6637
64,Sangat membantu,5154
28,Oke,2565
35,Baik,1945
120,Keren,1344
76,Top,1189
25,Nice,1141


[waktu] eda: 11 detik | total 0.7 menit


## 04 · Preprocessing dari universal

LF kata kunci butuh teks yang dinormalisasi, supaya "gak bagus" dan "ga bgs" terbaca sama. Model e5, NLI, dan IndoBERT memakai teks yang hanya dibersihkan secara teknis.

In [8]:
SLANG = dict(zip(
    'gk ga gak nggak ngga tdk blm jgn yg dgn utk krn karna bgt tp udh sdh skrg gmn klo kalo jg bgs'.split(),
    'tidak tidak tidak tidak tidak tidak belum jangan yang dengan untuk karena karena banget tapi sudah sudah sekarang bagaimana kalau kalau juga bagus'.split()))
PROTECTED = set('mbg sppg bgn apbn apbd bpom bpk umkm allah massa jakarta bandung'.split())
NEGATIONS = set('tidak tak bukan belum jangan tanpa kurang nggak gak ga gk ora teu indak'.split())
AMBIGUOUS = set('sm dr bs pd lg org'.split())
VOCAB = set('bagus banget lama makanan makan bergizi gratis program keracunan anggaran korupsi sehat sekolah anak siswa pemerintah merata enak'.split()) | set(SLANG.values()) | PROTECTED
STOPWORDS = set('yang dan di ke dari untuk pada dengan ini itu atau juga saya kami kita mereka dia oleh sebagai dalam para serta adalah sebuah'.split()) - NEGATIONS - PROTECTED
REGIONAL = {'ora': 'tidak', 'teu': 'tidak', 'indak': 'tidak', 'apik': 'bagus'}
OBFUSCATION = {'k3r4cunan': 'keracunan', 'k3r4cun4n': 'keracunan',
               'ker@cunan': 'keracunan', 'koru*si': 'korupsi'}
# URL/email/account/hashtag are single tokens: normalization never enters them.
TOKEN = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+|[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}|(?<!\w)[@#][\w.]+|[^\W\d_]+', re.I)
WORD = re.compile(r'[^\W\d_]+')
URL = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+', re.I)


def map_words(text, transform):
    return TOKEN.sub(lambda m: transform(m[0]) if WORD.fullmatch(m[0]) else m[0], text)

In [9]:
def valid_text(value):
    # None/NaN/pd.NA/numbers/list are not converted to literal "nan"/"123".
    return value if isinstance(value, str) else ''

def repair_encoding(text):
    return ftfy.fix_text(text, unescape_html=False, uncurl_quotes=False,
                        fix_latin_ligatures=False, fix_character_width=False,
                        remove_control_chars=False, normalization=None)

def normalize_unicode(text):
    return unicodedata.normalize('NFC', text)  # retain emoji/local scripts/fullwidth

def remove_html(text):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(text, 'html.parser')
    for tag in soup(['script', 'style']):
        tag.decompose()
    return soup.get_text(' ')

def decode_html_entities(text):
    return html.unescape(text)

def remove_invisible(text):
    text = text.translate(dict.fromkeys(map(ord, '\ufeff\u200b\u2060\x00'), None))
    return re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)

def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

In [10]:
def replace_urls(text):
    # Preserve terminal sentence punctuation; a heuristic, not a complete URL parser.
    return URL.sub(lambda m: ' URL ' + m[0][len(m[0].rstrip('.,!?;)')):], text)

def replace_emails(text):
    return re.sub(r'\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b', ' EMAIL ', text)

def replace_phones(text):
    return re.sub(r'(?<!\w)(?:\+62|62|0)[ -]?8(?:[ -]?\d){8,11}(?!\w)', ' PHONE ', text)

def replace_mentions(text):
    return re.sub(r'(?<![\w@])@[\w.]+', ' USER ', text)

def lowercase(text):
    return text.casefold()

def normalize_slang(text, mapping=SLANG):
    return map_words(text, lambda w: mapping.get(w.casefold(), w)
                     if w.casefold() not in PROTECTED | AMBIGUOUS else w)

def normalize_elongation(text, vocab=VOCAB):
    def shorten(word):
        if word.casefold() in PROTECTED:
            return word
        candidates = [re.sub(r'([^\W\d_])\1{2,}', replacement, word, flags=re.I)
                      for replacement in (r'\1', r'\1\1')]
        return next((w for w in candidates if w.casefold() in vocab), word)
    return map_words(text, shorten)

def normalize_laughter(text):
    def replace(word):
        return 'wkwk' if re.fullmatch(r'(?:wk){2,}|w+k+w+k+[wk]*', word, re.I) else (
            'haha' if re.fullmatch(r'(?:ha){2,}h?', word, re.I) else word)
    return map_words(text, replace)

def remove_punctuation(text):
    # Keep all scripts/letters/digits; punctuation becomes a SPACE, never concatenation.
    return re.sub(r'[^\w\s]|_', ' ', text)

In [11]:
def chain(text, steps):
    for step in steps: text = step(text)
    return text
def apply(series, steps):
    cache = {s: chain(s, steps) for s in series.unique()}; return series.map(cache)
SAFE = [valid_text, repair_encoding, normalize_unicode, lambda s: remove_html(s) if "<" in s else s, decode_html_entities, remove_invisible, normalize_whitespace]
with stage("preprocessing", len(U)):
    U["safe"] = apply(U.text, SAFE)
    U["embed"] = apply(U.safe, [replace_urls, replace_emails, replace_phones, replace_mentions, normalize_whitespace])
    U["normal"] = apply(U.embed, [lowercase, normalize_slang, normalize_elongation, normalize_laughter, remove_punctuation, normalize_whitespace])
show("Before/after", U[["text", "normal"]].sample(5, random_state=SEED))

[waktu] preprocessing: 24 detik | total 1.1 menit

Before/after


,text,normal
122728,Sangat puas selain banyak voucher banyak juga manfaatnya,sangat puas selain banyak voucher banyak juga manfaatnya
127975,Solusi disaat mager,solusi disaat mager
70261,food nya murah,food nya murah
117614,Saya suka aplikasi gojek Krn GK ribet dan byk promonya,saya suka aplikasi gojek karena tidak ribet dan byk promonya
24375,Semakin lemot aplikasinya tingkatkan,semakin lemot aplikasinya tingkatkan


## 05 · Labeling function

Setiap LF menghasilkan satu kolom di matriks `L`: indeks label, atau −1 kalau abstain. Leksikon ditulis eksplisit di sini supaya mudah diperiksa dan diganti untuk domain lain.

In [12]:
POS = set("""bagus baik mantap mantab mantul keren hebat puas memuaskan membantu terbantu cepat mudah gampang praktis ramah sopan
    nyaman aman murah hemat terbaik top oke ok good nice best great love suka senang sukses makasih terimakasih thanks lancar
    responsif profesional amanah enak sip jos joss recommended rekomen luar_biasa""".split()) | set(P["pos_extra"].split())
NEG = set("""jelek buruk parah payah kecewa mengecewakan lambat lama lemot lelet error eror bug ngebug crash gagal susah sulit ribet
    mahal rugi tipu penipu penipuan bohong kesal kesel marah benci sampah hancur rusak gangguan masalah komplain aneh menyesal
    nyesel bad worst lambat""".split()) | set(P["neg_extra"].split())
NEGATORS = {"tidak", "tak", "bukan", "belum", "kurang", "jangan", "enggak"}
QUESTION = {"bagaimana", "kenapa", "mengapa", "kapan", "apakah", "dimana", "gimana", "berapa", "bisakah", "apa"}
POS_EMOJI, NEG_EMOJI = set("😍👍🙏❤😊🥰👌💯✨😁😄🤩💪🥳"), set("😡😠😤👎😭😢💔🤬😒😞😩🙄")

def keyword_score(text):
    tokens = text.split(); score = 0
    for i, w in enumerate(tokens):
        polarity = 1 if w in POS else -1 if w in NEG else 0
        if polarity and NEGATORS & set(tokens[max(0, i - 2):i]): polarity = -polarity
        score += polarity
    return score

with stage("lf_murah", len(U)):
    score = U.normal.map(keyword_score).to_numpy()
    lf_keyword = np.where(score > 0, LID["positif"], np.where(score < 0, LID["negatif"], -1))
    asks = U.text.str.contains(r"\?") | U.normal.map(lambda s: bool(QUESTION & set(s.split()[:2])))
    lf_question = np.where(asks.to_numpy() & (score == 0), LID["netral"], -1) if "netral" in LID else np.full(len(U), -1)
    emo = U.text.map(lambda s: sum(c in POS_EMOJI for c in s) - sum(c in NEG_EMOJI for c in s)).to_numpy()
    lf_emoji = np.where(emo > 0, LID["positif"], np.where(emo < 0, LID["negatif"], -1))

[waktu] lf_murah: 2 detik | total 1.1 menit


In [13]:
def embed(model_id, texts, max_len=128, batch=BATCH):
    from transformers import AutoTokenizer, AutoModel
    tokenizer = AutoTokenizer.from_pretrained(model_id); model = AutoModel.from_pretrained(model_id).cuda().half().eval()
    order = np.argsort([len(x) for x in texts]); parts = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch):
            enc = tokenizer([texts[j] for j in order[i:i + batch]], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to("cuda")
            hidden = model(**enc).last_hidden_state; mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            parts.append(((hidden * mask).sum(1) / mask.sum(1)).float().cpu().numpy())
    vectors = np.zeros((len(texts), parts[0].shape[1]), "float32"); vectors[order] = np.concatenate(parts)
    del model; torch.cuda.empty_cache(); return normalize(vectors)

with stage("lf_e5", len(U)):
    E5_ID = locate("e5", "intfloat/multilingual-e5-base")
    E5 = embed(E5_ID, ("query: " + U.embed).tolist())
    desc = [("query: " + d, name) for name in LABELS for d in DESCRIPTIONS[name]]
    desc_vec = embed(E5_ID, [d for d, _ in desc], batch=16)
    protos = normalize(np.vstack([desc_vec[[i for i, (_, n) in enumerate(desc) if n == name]].mean(0) for name in LABELS]))
    sim = E5 @ protos.T; ordered = np.sort(sim, 1)
    lf_e5 = np.where(ordered[:, -1] - ordered[:, -2] >= E5_MARGIN, sim.argmax(1), -1)

[waktu] lf_e5: 77 detik | total 2.4 menit


In [14]:
with stage("lf_nli_sampel", N_NLI):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    nli_id = locate("mdeberta_nli", "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
    tok = AutoTokenizer.from_pretrained(nli_id); nli = AutoModelForSequenceClassification.from_pretrained(nli_id).cuda().eval()
    label2id = {k.lower(): v for k, v in nli.config.label2id.items()}; ent, con = label2id["entailment"], label2id["contradiction"]
    S_NLI = np.sort(rng.choice(np.where(~is_dev)[0], min(N_NLI, int((~is_dev).sum())), replace=False))
    premises = U.embed.to_numpy()[S_NLI]; order = np.argsort([len(p) for p in premises]); nli_prob = np.zeros((len(S_NLI), K), "float32")
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(order), 128):
            idx = order[i:i + 128]
            for j, name in enumerate(LABELS):
                enc = tok(list(premises[idx]), [HYPOTHESES[name]] * len(idx), truncation=True, max_length=128, padding=True, return_tensors="pt").to("cuda")
                logits = nli(**enc).logits.float()
                nli_prob[idx, j] = torch.softmax(logits[:, [ent, con]], 1)[:, 0].cpu().numpy()
    del nli; torch.cuda.empty_cache()
    nli_prob = nli_prob / nli_prob.sum(1, keepdims=True)
    lf_nli = np.full(len(U), -1); lf_nli[S_NLI] = np.where(nli_prob.max(1) >= NLI_MIN, nli_prob.argmax(1), -1)

[waktu] lf_nli_sampel: 81 detik | total 3.7 menit


In [15]:
L = np.column_stack([lf_keyword, lf_question, lf_emoji, lf_e5, lf_nli]); LF_NAMES = ["kata_kunci", "pertanyaan", "emoji", "e5_prototipe", "nli_zero_shot"]
covered = L >= 0
def lf_table(L, rows=slice(None)):
    out = []
    for j, name in enumerate(LF_NAMES):
        c = covered[rows, j]; w = W[rows]; others = np.delete(covered[rows], j, 1).any(1)
        Lr = L[rows]; rest = np.delete(Lr, j, 1)
        conflict = c & ((rest >= 0) & (rest != Lr[:, [j]])).any(1)
        dist = pd.Series(L[rows][c, j]).map(dict(enumerate(LABELS))).value_counts(normalize=True)
        out.append({"LF": name, "coverage_%_baris": 100 * w[c].sum() / w.sum(), "overlap_%": 100 * (c & others).mean(),
                    "conflict_%": 100 * conflict.mean(), **{f"porsi_{k}": dist.get(k, 0) for k in LABELS}})
    return pd.DataFrame(out)
show("5.1 Analisis LF di sampel NLI · coverage, overlap, conflict (tanpa label)", lf_table(L, S_NLI))
print(f"Baris tanpa satu LF pun di luar NLI: {100 * W[~covered[:, :4].any(1)].sum() / W.sum():.1f}%")


5.1 Analisis LF di sampel NLI · coverage, overlap, conflict (tanpa label)


,LF,coverage_%_baris,overlap_%,conflict_%,porsi_negatif,porsi_netral,porsi_positif
0,kata_kunci,61.120,58.820,9.695,0.337,0.000,0.663
1,pertanyaan,1.754,2.195,2.195,0.000,1.000,0.000
2,emoji,0.000,0.000,0.000,0.000,0.000,0.000
3,e5_prototipe,38.378,32.580,1.285,0.209,0.006,0.786
4,nli_zero_shot,98.968,66.685,12.275,0.450,0.001,0.549


Baris tanpa satu LF pun di luar NLI: 25.2%


## 06 · Label model

Dua cara menggabungkan LF.
- **Majority vote:** suara terbanyak. Kalau seri, baris itu abstain.
- **Weighted vote:** akurasi setiap LF diperkirakan dari kesepakatannya dengan suara LF lain (leave-one-out) di baris yang tersentuh minimal dua LF, lalu dijadikan bobot log-odds. Kalau ada dev berlabel, akurasi diambil langsung dari dev.

Hasilnya berupa probabilitas kelas. Baris dengan keyakinan ≥ `CONF_MIN` menjadi data latih end model.

In [16]:
def vote(L, weights):
    score = np.zeros((len(L), K))
    for j, w in enumerate(weights):
        hit = L[:, j] >= 0; score[np.where(hit)[0], L[hit, j]] += w
    prob = np.exp(score - score.max(1, keepdims=True)); prob[score.sum(1) == 0] = 1 / K
    return prob / prob.sum(1, keepdims=True), score.sum(1) > 0

mv_prob, mv_any = vote(L, np.ones(L.shape[1]))
mv = np.where(mv_any & (np.sort(mv_prob, 1)[:, -1] > np.sort(mv_prob, 1)[:, -2]), mv_prob.argmax(1), -1)
acc = []
for j in range(L.shape[1]):
    others, _ = vote(np.delete(L, j, 1), np.ones(L.shape[1] - 1))
    rows = covered[:, j] & np.delete(covered, j, 1).any(1)
    if is_dev.any() and (covered[:, j] & is_dev).sum() >= 20:
        truth = U["_audit"].map(LID).to_numpy(); rows_dev = covered[:, j] & is_dev
        acc.append(float((L[rows_dev, j] == truth[rows_dev]).mean()))
    else:
        acc.append(float((L[rows, j] == others[rows].argmax(1)).mean()) if rows.any() else 1 / K)
acc = np.clip(np.array(acc), 1 / K + 1e-3, .99)
weights = np.log(acc * (K - 1) / (1 - acc))
lm_prob, lm_any = vote(L, weights); lm = np.where(lm_any, lm_prob.argmax(1), -1)
show("6.1 Bobot LF", pd.DataFrame({"LF": LF_NAMES, "akurasi_perkiraan": acc, "bobot": weights, "sumber": "dev" if is_dev.any() else "kesepakatan antar LF"}))
train_mask = (lm >= 0) & (lm_prob.max(1) >= CONF_MIN) & ~is_dev
show("6.2 Data latih dari label lemah", pd.Series({"teks_unik_berlabel_lemah": int(train_mask.sum()), "coverage_%_baris": 100 * W[lm >= 0].sum() / W.sum(),
    **{f"porsi_{k}": (lm[train_mask] == i).mean() for i, k in enumerate(LABELS)}}))


6.1 Bobot LF


,LF,akurasi_perkiraan,bobot,sumber
0,kata_kunci,0.796,2.053,dev
1,pertanyaan,0.334,0.004,dev
2,emoji,0.334,0.004,dev
3,e5_prototipe,0.941,3.459,dev
4,nli_zero_shot,0.823,2.229,dev



6.2 Data latih dari label lemah


,0
teks_unik_berlabel_lemah,92206.000
coverage_%_baris,78.152
porsi_negatif,0.355
porsi_netral,0.003
porsi_positif,0.642


## 07 · End model

Lima pendekatan dilatih dari label lemah yang sama, ditambah ensemble rata-rata tanpa bobot. Kelas diberi bobot seimbang, dan baris berlabel lemah diberi bobot sebesar keyakinan label model. IndoBERT dilatih di sampel `N_BERT` yang seimbang per kelas.

In [17]:
y_weak, conf = lm[train_mask], lm_prob[train_mask].max(1); PROB = {}
majority = np.bincount(y_weak, minlength=K).argmax()
PROB["01_mayoritas"] = np.eye(K)[np.full(len(U), majority)]
PROB["02_kata_kunci_saja"] = np.eye(K)[np.where(lf_keyword >= 0, lf_keyword, majority)]
with stage("tfidf_lr", len(U)):
    tfidf = TfidfVectorizer(min_df=3, ngram_range=(1, 2), max_features=200000, sublinear_tf=True).fit(U.normal)
    X = tfidf.transform(U.normal)
    PROB["03_tfidf_lr"] = LogisticRegression(C=2, max_iter=2000, class_weight="balanced").fit(X[train_mask], y_weak, sample_weight=conf).predict_proba(X)
with stage("e5_lr", len(U)):
    PROB["04_e5_lr"] = LogisticRegression(C=2, max_iter=2000, class_weight="balanced").fit(E5[train_mask], y_weak, sample_weight=conf).predict_proba(E5)

[waktu] tfidf_lr: 22 detik | total 4.1 menit


[waktu] e5_lr: 12 detik | total 4.3 menit


In [18]:
with stage("indobert_finetune", N_BERT):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    bert_id = locate("indobert", "indobenchmark/indobert-base-p1")
    tok = AutoTokenizer.from_pretrained(bert_id)
    model = AutoModelForSequenceClassification.from_pretrained(bert_id, num_labels=K, ignore_mismatched_sizes=True).cuda()
    pool = np.where(train_mask)[0]; per_class = N_BERT // K
    pick = np.concatenate([rng.permutation(pool[lm[pool] == c])[:per_class] for c in range(K)]); rng.shuffle(pick)
    texts, target = U.embed.str.lower().to_numpy(), torch.tensor(lm)
    class_w = torch.tensor(len(pick) / (K * np.maximum(np.bincount(lm[pick], minlength=K), 1)), dtype=torch.float).cuda()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=.01); bs = 64
    steps = BERT_EPOCHS * int(np.ceil(len(pick) / bs)); scheduler = get_linear_schedule_with_warmup(optimizer, int(.1 * steps), steps)
    scaler = torch.amp.GradScaler("cuda"); model.train()
    for epoch in range(BERT_EPOCHS):
        for i in range(0, len(pick), bs):
            idx = pick[i:i + bs]; enc = tok(list(texts[idx]), truncation=True, max_length=BERT_LEN, padding=True, return_tensors="pt").to("cuda")
            with torch.autocast("cuda", dtype=torch.float16):
                loss = torch.nn.functional.cross_entropy(model(**enc).logits.float(), target[idx].cuda(), weight=class_w)
            optimizer.zero_grad(); scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
    model.eval(); order = np.argsort([len(x) for x in texts]); bert_prob = np.zeros((len(U), K), "float32")
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(order), BATCH):
            idx = order[i:i + BATCH]; enc = tok(list(texts[idx]), truncation=True, max_length=BERT_LEN, padding=True, return_tensors="pt").to("cuda")
            bert_prob[idx] = torch.softmax(model(**enc).logits.float(), 1).cpu().numpy()
    del model; torch.cuda.empty_cache()
PROB["05_indobert_weak"] = bert_prob
PROB["06_ensemble_rata"] = (PROB["03_tfidf_lr"] + PROB["04_e5_lr"] + PROB["05_indobert_weak"]) / 3
print(f"IndoBERT dilatih di {len(pick)} teks berlabel lemah, {BERT_EPOCHS} epoch")

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


[transformers] BertForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/ivanwllm/gemastik-nlp-00-setup-offline/models/indobert
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[waktu] indobert_finetune: 187 detik | total 7.4 menit
IndoBERT dilatih di 20292 teks berlabel lemah, 2 epoch


## 08 · Pemilihan & post-processing

**Tanpa dev:** ensemble rata-rata dipakai sebagai default, karena tiga representasi yang berbeda membuat kesalahan masing-masing saling meredam. Tabel kesepakatan antar model hanya ditampilkan, tidak dipakai memilih.

**Dengan dev:** model dengan macro-F1 dev tertinggi dipilih. Lalu pengali kelas dicoba secara cross-fit di dev, dan dipakai hanya kalau naik minimal 0,005.

**Konsistensi duplikat** terjadi otomatis, karena prediksi dibuat per teks unik.

In [19]:
names = list(PROB)
agree = pd.DataFrame([[np.average(PROB[a].argmax(1) == PROB[b].argmax(1), weights=W) for b in names] for a in names], index=names, columns=names)
show("8.1 Kesepakatan antar model · % baris (tanpa label)", 100 * agree)
show("Distribusi prediksi · % baris", pd.DataFrame({n: pd.Series(np.bincount(PROB[n].argmax(1), weights=W, minlength=K) / W.sum(), index=LABELS) for n in names}).T * 100)
def macro(y, p, w=None): return f1_score(y, p, labels=range(K), average="macro", zero_division=0, sample_weight=w)
multipliers = np.ones(K)
if is_dev.any():
    y_dev = U["_audit"].map(LID).to_numpy()[DEV]
    dev_scores = pd.Series({n: macro(y_dev, PROB[n][DEV].argmax(1), W[DEV]) for n in names}).sort_values(ascending=False)
    show("8.2 Macro-F1 di dev berlabel", dev_scores)
    FINAL = dev_scores.index[0]
    def fit_mult(p, y, w):
        m = np.ones(K)
        for _ in range(2):
            for c in range(K):
                m[c] = max([.5, .75, 1, 1.25, 1.5, 2], key=lambda v: macro(y, (p * np.where(np.arange(K) == c, v, m)).argmax(1), w))
        return m
    p_dev, cross = PROB[FINAL][DEV], np.zeros(len(DEV), int)
    for a, b in StratifiedKFold(2, shuffle=True, random_state=SEED).split(p_dev, y_dev):
        cross[b] = (p_dev[b] * fit_mult(p_dev[a], y_dev[a], W[DEV][a])).argmax(1)
    gain = macro(y_dev, cross, W[DEV]) - macro(y_dev, p_dev.argmax(1), W[DEV])
    if gain >= .005: multipliers = fit_mult(p_dev, y_dev, W[DEV])
    print(f"Pengali kelas cross-fit: kenaikan {gain:+.4f} -> {'dipakai ' + str(multipliers) if gain >= .005 else 'tidak dipakai'}")
else:
    FINAL = "06_ensemble_rata"
final_prob = PROB[FINAL] * multipliers; final_prob /= final_prob.sum(1, keepdims=True); final = final_prob.argmax(1)
print("FINAL:", FINAL)


8.1 Kesepakatan antar model · % baris (tanpa label)


,01_mayoritas,02_kata_kunci_saja,03_tfidf_lr,04_e5_lr,05_indobert_weak,06_ensemble_rata
01_mayoritas,100.000,87.467,68.672,71.202,71.015,71.994
02_kata_kunci_saja,87.467,100.000,81.013,82.271,83.017,84.162
03_tfidf_lr,68.672,81.013,100.000,86.795,87.323,90.695
04_e5_lr,71.202,82.271,86.795,100.000,90.871,94.372
05_indobert_weak,71.015,83.017,87.323,90.871,100.000,95.570
06_ensemble_rata,71.994,84.162,90.695,94.372,95.570,100.000



Distribusi prediksi · % baris


,negatif,netral,positif
01_mayoritas,0.000,0.000,100.000
02_kata_kunci_saja,12.533,0.000,87.467
03_tfidf_lr,25.796,5.532,68.672
04_e5_lr,26.507,2.291,71.202
05_indobert_weak,25.114,3.871,71.015
06_ensemble_rata,26.000,2.006,71.994



8.2 Macro-F1 di dev berlabel


,0
04_e5_lr,0.582
06_ensemble_rata,0.575
05_indobert_weak,0.567
03_tfidf_lr,0.563
02_kata_kunci_saja,0.468
01_mayoritas,0.261


Pengali kelas cross-fit: kenaikan +0.0123 -> dipakai [1.5  1.25 0.75]
FINAL: 04_e5_lr


## 09 · Evaluasi & kelemahan (audit)

Label audit dibuka di sini. Baris dev dikeluarkan dari audit. Semua skor berbobot jumlah baris, jadi setara skor per baris asli. Rating sebagai kebenaran juga berisik: tabel 9.6 menunjukkan contohnya.

In [20]:
truth = U["_audit"].map(LID).to_numpy(); evaluable = ~is_dev & ~pd.isna(truth)
y, w = truth[evaluable].astype(int), W[evaluable]
show("9.0 Distribusi label audit · % baris", pd.Series(np.bincount(y, weights=w, minlength=K) / w.sum() * 100, index=LABELS))
rows = []
for j, name in enumerate(LF_NAMES):
    c = covered[evaluable, j]
    rows.append({"sumber": "LF " + name, "coverage_%": 100 * w[c].sum() / w.sum(), "akurasi_saat_aktif": np.average(L[evaluable][c, j] == y[c], weights=w[c]) if c.any() else np.nan,
                 "macro_f1_saat_aktif": macro(y[c], L[evaluable][c, j], w[c]) if c.any() else np.nan})
for name, labels in [("label model: majority", mv), ("label model: weighted", lm)]:
    c = labels[evaluable] >= 0
    rows.append({"sumber": name, "coverage_%": 100 * w[c].sum() / w.sum(), "akurasi_saat_aktif": np.average(labels[evaluable][c] == y[c], weights=w[c]),
                 "macro_f1_saat_aktif": macro(y[c], labels[evaluable][c], w[c])})
show("9.1 LF dan label model · hanya di baris yang mereka beri label", pd.DataFrame(rows))
models = pd.DataFrame([{"model": n, "macro_f1": macro(y, PROB[n][evaluable].argmax(1), w), "accuracy": np.average(PROB[n][evaluable].argmax(1) == y, weights=w),
    **{f"f1_{k}": f1_score(y, PROB[n][evaluable].argmax(1), labels=[i], average="macro", zero_division=0, sample_weight=w) for i, k in enumerate(LABELS)}} for n in names])
models.loc[len(models)] = {"model": f"FINAL ({FINAL})", "macro_f1": macro(y, final[evaluable], w), "accuracy": np.average(final[evaluable] == y, weights=w),
    **{f"f1_{k}": f1_score(y, final[evaluable], labels=[i], average="macro", zero_division=0, sample_weight=w) for i, k in enumerate(LABELS)}}
show("9.2 End model · semua baris non-dev", models)
nli_rows = np.intersect1d(S_NLI, np.where(evaluable)[0]); yn = truth[nli_rows].astype(int)
print(f"Zero-shot NLI langsung di sampel {len(nli_rows)}: macro-F1 {macro(yn, nli_prob[np.searchsorted(S_NLI, nli_rows)].argmax(1), W[nli_rows]):.3f}"
      f" | FINAL di sampel yang sama: {macro(yn, final[nli_rows], W[nli_rows]):.3f}")


9.0 Distribusi label audit · % baris


,0
negatif,23.948
netral,3.760
positif,72.292



9.1 LF dan label model · hanya di baris yang mereka beri label


,sumber,coverage_%,akurasi_saat_aktif,macro_f1_saat_aktif
0,LF kata_kunci,68.751,0.897,0.568
1,LF pertanyaan,1.271,0.147,0.085
2,LF emoji,0.000,NaN,NaN
3,LF e5_prototipe,33.448,0.966,0.639
4,LF nli_zero_shot,11.247,0.823,0.549
5,label model: majority,76.862,0.889,0.600
6,label model: weighted,78.204,0.884,0.598



9.2 End model · semua baris non-dev


,model,macro_f1,accuracy,f1_negatif,f1_netral,f1_positif
0,01_mayoritas,0.280,0.723,0.000,0.000,0.839
1,02_kata_kunci_saja,0.476,0.806,0.542,0.000,0.885
2,03_tfidf_lr,0.559,0.819,0.761,0.023,0.892
3,04_e5_lr,0.582,0.857,0.762,0.061,0.923
4,05_indobert_weak,0.591,0.852,0.760,0.088,0.924
5,06_ensemble_rata,0.592,0.868,0.778,0.067,0.932
6,FINAL (04_e5_lr),0.583,0.851,0.770,0.061,0.919


Zero-shot NLI langsung di sampel 20000: macro-F1 0.547 | FINAL di sampel yang sama: 0.571


In [21]:
cm = pd.DataFrame(confusion_matrix(y, final[evaluable], labels=range(K), sample_weight=w), index=LABELS, columns=LABELS)
show("9.3 Confusion FINAL · % per kelas audit (baris=asli, kolom=prediksi)", cm.div(cm.sum(1), axis=0) * 100)
frame = pd.DataFrame({"kata": U.text.str.split().str.len().to_numpy()[evaluable], "y": y, "p": final[evaluable], "w": w})
for name, flag in flags.items(): frame[name] = flag.to_numpy()[evaluable]
frame["tanpa_LF_murah"] = ~covered[evaluable, :4].any(1)
frame["panjang"] = pd.cut(frame.kata, [0, 2, 5, 15, 40, np.inf], labels=["1-2", "3-5", "6-15", "16-40", ">40"])
slices = []
for column in ["panjang", "negasi", "tanda_tanya", "emoji", "kata_inggris", "slang", "tanpa_LF_murah"]:
    for value, part_ in frame.groupby(column, observed=True):
        slices.append({"slice": column, "nilai": str(value), "baris_%": 100 * part_.w.sum() / w.sum(), "macro_f1": macro(part_.y, part_.p, part_.w),
                       "error_%": 100 * np.average(part_.y != part_.p, weights=part_.w)})
show("9.4 Di mana model lemah", pd.DataFrame(slices))


9.3 Confusion FINAL · % per kelas audit (baris=asli, kolom=prediksi)


,negatif,netral,positif
negatif,86.247,3.669,10.084
netral,72.791,5.326,21.884
positif,8.774,2.344,88.883



9.4 Di mana model lemah


,slice,nilai,baris_%,macro_f1,error_%
0,panjang,1-2,43.034,0.529,6.505
1,panjang,3-5,18.810,0.556,14.484
2,panjang,6-15,21.523,0.555,23.178
3,panjang,16-40,13.359,0.503,26.380
4,panjang,>40,3.274,0.462,26.073
5,negasi,False,83.901,0.572,12.227
6,negasi,True,16.099,0.472,28.768
7,tanda_tanya,False,100.000,0.583,14.890
8,emoji,False,100.000,0.583,14.890
9,kata_inggris,False,93.466,0.581,15.814


In [22]:
analysis = U.loc[evaluable, ["text", "count"]].assign(audit=[LABELS[i] for i in y], prediksi=[LABELS[i] for i in final[evaluable]],
                                                       yakin=final_prob[evaluable].max(1), skor_kata_kunci=score[evaluable])
wrong = analysis[analysis.audit != analysis.prediksi]
show("9.5 Salah dengan keyakinan tertinggi · sering rating yang tidak cocok dengan isinya", wrong.nlargest(8, "yakin"))
show("9.6 Salah yang paling sering muncul (berbobot baris)", wrong.nlargest(8, "count"))
show("9.7 Netral menurut audit, ditebak apa?", analysis[analysis.audit.eq("netral")].prediksi.value_counts(normalize=True) * 100)
weakest = models.iloc[-1][[f"f1_{k}" for k in LABELS]].astype(float).idxmin()
print(f"Kelas terlemah FINAL: {weakest} | macro-F1 FINAL {models.iloc[-1].macro_f1:.3f} | terbaik di tabel 9.2: {models.sort_values('macro_f1').iloc[-1].model}")


9.5 Salah dengan keyakinan tertinggi · sering rating yang tidak cocok dengan isinya


,text,count,audit,prediksi,yakin,skor_kata_kunci
10790,apa ada customer service untuk bertanya,1,negatif,netral,1.0,0
103401,Yes good,1,netral,positif,1.0,1
118802,Kecewa Skg lemot susah masuk ny,1,netral,negatif,1.0,-3
84697,Lagi butuh token listrik malah susah banget beli nya error nya lama,1,positif,negatif,1.0,-3
129234,cukup membantu dalam mengantar penumpang dgn baik,1,netral,positif,1.0,2
65772,Cukup baik driver nya,1,netral,positif,1.0,2
90608,Claim buat pesanan yg diterima tidak sesuai diprosesnya lama bgt,1,positif,negatif,1.0,-1
61343,SULIT DI MINTA INFORMASI NO DRIVER MENGENAI PERIHAL PENIPUAN,1,negatif,netral,1.0,-1



9.6 Salah yang paling sering muncul (berbobot baris)


,text,count,audit,prediksi,yakin,skor_kata_kunci
595,Memuaskan,556,positif,netral,0.566,1
1557,Siip,281,positif,netral,0.621,0
1597,Lumayan,265,positif,negatif,0.704,0
1163,Ramah,134,positif,negatif,0.677,1
5471,Siiip,111,positif,netral,0.719,0
5586,Gud,77,positif,negatif,0.677,0
2954,Hebat,59,positif,negatif,0.505,1
493,Kren,46,positif,negatif,0.766,0



9.7 Netral menurut audit, ditebak apa?


,proportion
prediksi,
negatif,73.293
positif,21.562
netral,5.144


Kelas terlemah FINAL: f1_netral | macro-F1 FINAL 0.583 | terbaik di tabel 9.2: 06_ensemble_rata


## 10 · Keluaran & waktu

`prediksi.csv` berisi label untuk setiap baris asli. Waktu tiap tahap dicatat, dan notebook dibuat gagal kalau total melewati `BUDGET_MINUTES`.

In [23]:
label_of = dict(zip(U.key, [LABELS[i] for i in final])); conf_of = dict(zip(U.key, final_prob.max(1)))
data[["row_id", TEXT_COL]].assign(prediksi=data.key.map(label_of), yakin=data.key.map(conf_of)).to_csv(OUT / "prediksi.csv", index=False)
models.to_csv(OUT / "end_model.csv", index=False); pd.DataFrame(slices).to_csv(OUT / "slices.csv", index=False)
times = pd.DataFrame(TIMES); times["menit"] = times.detik / 60; show("Waktu per tahap", times)
total = (time.perf_counter() - RUN_START) / 60
import importlib.metadata as metadata
manifest = {"dataset": DATASET, "n_dev": N_DEV, "rows": len(data), "unique": len(U), "final": FINAL, "multipliers": multipliers.tolist(),
    "lf_weights": dict(zip(LF_NAMES, weights.tolist())), "audit": models.iloc[-1].to_dict(), "minutes": total, "times": TIMES,
    "gpu": torch.cuda.get_device_name(0), "versions": {p: metadata.version(p) for p in ["torch", "transformers", "scikit-learn", "pandas"]}}
(OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=float))
print(f"Total {total:.1f} menit | batas {BUDGET_MINUTES} menit")
assert total < BUDGET_MINUTES


Waktu per tahap


,tahap,detik,n,menit
0,muat_data,0.593,NaN,0.010
1,eda,11.134,134606.0,0.186
2,preprocessing,24.190,134606.0,0.403
3,lf_murah,1.896,134606.0,0.032
4,lf_e5,76.857,134606.0,1.281
5,lf_nli_sampel,81.128,20000.0,1.352
6,tfidf_lr,22.154,134606.0,0.369
7,e5_lr,12.323,134606.0,0.205
8,indobert_finetune,187.294,30000.0,3.122


Total 7.5 menit | batas 290 menit


## Batas interpretasi
- Leksikon, deskripsi label, dan hipotesis NLI ditulis manual untuk ulasan aplikasi. Untuk MBG, ganti `pos_extra`, `neg_extra`, dan deskripsinya, lalu periksa tabel 5.1 sebelum mempercayai hasil.
- Bobot label model tanpa dev diperkirakan dari kesepakatan antar LF. LF yang salah dengan cara yang sama, misalnya e5 dan NLI yang sama-sama terkecoh sarkasme, akan saling menguatkan.
- Rating bintang bukan label sentimen sempurna, jadi skor audit adalah batas bawah kasar.
- Satu seed. IndoBERT hanya dilatih 2 epoch di sampel. Hasilnya tidak membuktikan model mana yang selalu unggul.